In [17]:
import pandas as pd

In [18]:
df = pd.read_csv('100_Unique_QA_Dataset.csv')

In [19]:
def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'", "")

    return text.split()

In [20]:
tokenize("what is 'captial' of france?")

['what', 'is', 'captial', 'of', 'france']

In [21]:
vocab = {'<UNK>':0}

In [22]:
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    merged_token = tokenized_question + tokenized_answer

    for token in merged_token:
        if token not in vocab:
            vocab[token] = len(vocab)

In [23]:
df.apply(build_vocab, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [24]:
len(vocab)

324

In [25]:
def text_to_indices(text, vocab):
    indexed_text = []
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text

In [26]:
import torch
from torch.utils.data import Dataset, DataLoader

In [27]:
class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

        return torch.tensor(numerical_question), torch.tensor(numerical_answer)


In [28]:
dataset = QADataset(df, vocab)

In [29]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [30]:
import torch.nn as nn

In [31]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))

        return output


In [32]:
learning_rate = 0.001
epochs = 20

In [33]:
model = SimpleRNN(len(vocab))

In [34]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [36]:
for epoch in range(epochs):
    total_loss = 0

    for question, answer in dataloader:
        optimizer.zero_grad()

        output = model(question)

        loss = criterion(output, answer[0])

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 522.397501
Epoch: 2, Loss: 452.520175
Epoch: 3, Loss: 375.157403
Epoch: 4, Loss: 316.277843
Epoch: 5, Loss: 266.941374
Epoch: 6, Loss: 219.757624
Epoch: 7, Loss: 176.724031
Epoch: 8, Loss: 138.807297
Epoch: 9, Loss: 107.258547
Epoch: 10, Loss: 82.557508
Epoch: 11, Loss: 63.638796
Epoch: 12, Loss: 50.063170
Epoch: 13, Loss: 39.859641
Epoch: 14, Loss: 32.137080
Epoch: 15, Loss: 26.664256
Epoch: 16, Loss: 22.113768
Epoch: 17, Loss: 18.454226
Epoch: 18, Loss: 15.616520
Epoch: 19, Loss: 13.269834
Epoch: 20, Loss: 11.493690


In [37]:
def predict(model, question, threshold=0.5):
    numerical_question = text_to_indices(question, vocab)

    question_tensor = torch.tensor(numerical_question).unsqueeze(0)

    output = model(question_tensor)
    probs = torch.nn.functional.softmax(output, dim=1)

    value, index = torch.max(probs, dim=1)

    if value < threshold:
        print("I don't know")

    print(list(vocab.keys())[index])

In [48]:
predict(model, 'king of fruits')

mango
